In [1]:
import tensorflow as tf
import numpy as np

# 1. Load dataset CIFAR-10 langsung dari internet lewat Keras
(x_train_full, y_train_full), (x_test_full, y_test_full) = tf.keras.datasets.cifar10.load_data()

# 2. Saring agar hanya mengambil 2 kelas saja: Kucing (label 3) dan Anjing (label 5)
# Kamu bebas ganti kelas lain, tapi ini biar pas dengan tema tugasmu
keep_train = np.where((y_train_full == 3) | (y_train_full == 5))[0]
keep_test = np.where((y_test_full == 3) | (y_test_full == 5))[0]

x_train_filtered = x_train_full[keep_train]
y_train_filtered = y_train_full[keep_train]
x_test_filtered = x_test_full[keep_test]
y_test_filtered = y_test_full[keep_test]

# Ubah label menjadi biner: Kucing = 0, Anjing = 1
y_train_filtered = np.where(y_train_filtered == 3, 0, 1).astype('float32')
y_test_filtered = np.where(y_test_filtered == 3, 0, 1).astype('float32')

# 3. Bagi data test menjadi Validation (15%) dan Testing (15%) sesuai aturan tugas
from sklearn.model_selection import train_test_split
x_val, x_test, y_val, y_test = train_test_split(
    x_test_filtered, y_test_filtered, test_size=0.5, random_state=42
)

print(f"Data Training   : {x_train_filtered.shape[0]} gambar")
print(f"Data Validation : {x_val.shape[0]} gambar")
print(f"Data Testing    : {x_test.shape[0]} gambar")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Data Training   : 10000 gambar
Data Validation : 1000 gambar
Data Testing    : 1000 gambar


In [2]:
from tensorflow.keras import layers, models

model_scratch = models.Sequential([
    # Input disesuaikan dengan ukuran CIFAR-10 yaitu 32x32 piksel
    layers.Input(shape=(32, 32, 3)),
    layers.Rescaling(1./255),

    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Flatten & Classifier
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),

    # Output layer (Binary Classification)
    layers.Dense(1, activation='sigmoid')
])

model_scratch.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [3]:
EPOCHS = 20
BATCH_SIZE = 32

history_scratch = model_scratch.fit(
    x_train_filtered, y_train_filtered,
    validation_data=(x_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

Epoch 1/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 17s 43ms/step - accuracy: 0.5986 - loss: 0.7706 - val_accuracy: 0.5370 - val_loss: 1.5773
Epoch 2/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 13s 42ms/step - accuracy: 0.6495 - loss: 0.6397 - val_accuracy: 0.6320 - val_loss: 0.6841
Epoch 3/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 13s 42ms/step - accuracy: 0.6860 - loss: 0.5892 - val_accuracy: 0.6350 - val_loss: 0.6427
Epoch 4/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 14s 46ms/step - accuracy: 0.7094 - loss: 0.5613 - val_accuracy: 0.5690 - val_loss: 0.7984
Epoch 5/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 14s 43ms/step - accuracy: 0.7266 - loss: 0.5420 - val_accuracy: 0.6860 - val_loss: 0.5860
Epoch 6/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 13s 42ms/step - accuracy: 0.7283 - loss: 0.5334 - val_accuracy: 0.7180 - val_loss: 0.5802
Epoch 7/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 13s 42ms/step - accuracy: 0.7395 - loss: 0.5177 - val_accuracy: 0.7170 - val_loss: 0.5474
Epoch 8/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 14s 43ms/step - accuracy: 0.7535 - loss: 0.4981 - 

In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Load Pretrained Model MobileNetV2 (Kembali ke kode awal yang aman)
base_model_cifar = tf.keras.applications.MobileNetV2(
    input_shape=(32, 32, 3),
    include_top=False,
    weights='imagenet'
)

# 2. Bekukan seluruh layer pretrained (Feature Extraction)
base_model_cifar.trainable = False

# 3. Bangun Top Classifier Baru
model_tl_cifar = models.Sequential([
    base_model_cifar,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid') # Biner: Kucing vs Anjing
])

# 4. Compile Model
model_tl_cifar.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("✅ Model Transfer Learning MobileNetV2 untuk CIFAR-10 siap!")

/tmp/ipykernel_3599/1421604359.py:5: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model_cifar = tf.keras.applications.MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
✅ Model Transfer Learning MobileNetV2 untuk CIFAR-10 siap!


In [5]:
EPOCHS_TL = 10
BATCH_SIZE = 32

history_tl = model_tl_cifar.fit(
    x_train_filtered, y_train_filtered,
    validation_data=(x_val, y_val),
    epochs=EPOCHS_TL,
    batch_size=BATCH_SIZE
)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 24ms/step - accuracy: 0.5308 - loss: 0.6898 - val_accuracy: 0.5520 - val_loss: 0.6867
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - accuracy: 0.5396 - loss: 0.6885 - val_accuracy: 0.5550 - val_loss: 0.6854
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.5441 - loss: 0.6870 - val_accuracy: 0.5370 - val_loss: 0.6848
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - accuracy: 0.5438 - loss: 0.6864 - val_accuracy: 0.5630 - val_loss: 0.6834
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.5415 - loss: 0.6856 - val_accuracy: 0.5550 - val_loss: 0.6832
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step - accuracy: 0.5440 - loss: 0.6852 - val_accuracy: 0.5530 - val_loss: 0.6826
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.5444 - loss: 0.6848 - val_accuracy: 0.5530 - val_loss: 0.6823
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 20ms/step - accuracy: 0.5422 - loss: 0.6840 - val_ac